In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
sns.set_theme()

In [ ]:
df1 = pd.read_csv(
    r'C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\Etapa3\acidentes_pbic_2020_2025_Final.csv',
    encoding='utf-8',
    parse_dates=['data_inversa'],
    dayfirst=True,
    low_memory=False
)

print("Shape inicial:", df1.shape)


Shape inicial: (1678326, 48)


In [3]:
df = pd.read_csv(
    r'C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\acidentes_pbic_2020_2025_limpo.csv',
    encoding='utf-8',
    parse_dates=['data_inversa'],
    dayfirst=True,
    low_memory=False
)

print("Shape inicial:", df.shape)


Shape inicial: (2782541, 43)


In [4]:
import pandas as pd

# =====================================
# 1. Criar cópia
# =====================================
df_novo = df.copy()

# =====================================
# 2. Tratar valores nulos
# =====================================
df_novo["pesid"] = df_novo["pesid"].fillna(-1)
df_novo["marca"] = df_novo["marca"].fillna("DESCONHECIDO")

# =====================================
# 3. Garantir 1 linha por pessoa no acidente
# (evita duplicar vítimas)
# =====================================
df_pessoa = (
    df_novo
    .groupby(["id", "pesid"], as_index=False)
    .agg({
        "mortos": "max",
        "feridos_graves": "max",
        "feridos_leves": "max",
        "ilesos": "max",
        "marca": "first"
    })
)

# =====================================
# 4. 🔥 SOMA POR (id + marca)
# =====================================
df_soma_marca = (
    df_pessoa
    .groupby(["id", "marca"], as_index=False)
    .agg({
        "mortos": "sum",
        "feridos_graves": "sum",
        "feridos_leves": "sum",
        "ilesos": "sum"
    })
    .rename(columns={
        "mortos": "Sum_Mortos",
        "feridos_graves": "Sum_Feridos_Graves",
        "feridos_leves": "Sum_Feridos_Leves",
        "ilesos": "Sum_Ilesos"
    })
)

# =====================================
# 5. Total de vítimas por (id + marca)
# =====================================
df_soma_marca["Sum_Total_Vitimas"] = (
    df_soma_marca["Sum_Mortos"] +
    df_soma_marca["Sum_Feridos_Graves"] +
    df_soma_marca["Sum_Feridos_Leves"]
)

# =====================================
# 6. Trazer de volta pro dataset original
# (sem perder nenhuma linha)
# =====================================
df_novo = df_novo.merge(
    df_soma_marca,
    on=["id", "marca"],
    how="left"
)

# =====================================
# 7. Salvar nova base
# =====================================
df_novo.to_csv(
    r"C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\acidentes_por_marca.csv",
    index=False
)

# =====================================
# 8. Conferência
# =====================================
print(df_novo.head())

print(df_novo[[
    "id",
    "marca",
    "pesid",
    "Sum_Mortos",
    "Sum_Feridos_Graves",
    "Sum_Feridos_Leves",
    "Sum_Total_Vitimas"
]].head(20))

         id     pesid data_inversa    dia_semana   horario  uf     br     km  \
0  260031.0  578988.0   2020-01-01  quarta-feira  01:00:00  TO  153.0  678,1   
1  260031.0  578987.0   2020-01-01  quarta-feira  01:00:00  TO  153.0  678,1   
2  260031.0  578991.0   2020-01-01  quarta-feira  01:00:00  TO  153.0  678,1   
3  260031.0  578986.0   2020-01-01  quarta-feira  01:00:00  TO  153.0  678,1   
4  260031.0  578475.0   2020-01-01  quarta-feira  01:00:00  TO  153.0  678,1   

  municipio causa_principal  ... mes  dia_mes hora gravidade_numerica  \
0    GURUPI             Sim  ...   1        1    1                  2   
1    GURUPI             Sim  ...   1        1    1                  2   
2    GURUPI             Sim  ...   1        1    1                  2   
3    GURUPI             Sim  ...   1        1    1                  2   
4    GURUPI             Sim  ...   1        1    1                  2   

  total_vitimas Sum_Mortos Sum_Feridos_Graves Sum_Feridos_Leves Sum_Ilesos  \
0 

In [5]:
df2 = pd.read_csv(
    r'C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\acidentes_por_marca.csv',
    encoding='utf-8',
    parse_dates=['data_inversa'],
    dayfirst=True,
    low_memory=False
)

print("Shape inicial:", df2.shape)

Shape inicial: (2782541, 48)


In [6]:
# import pandas as pd

# # =====================================
# # 1. Criar cópia (mantém original intacto)
# # =====================================
# df_novo = df.copy()

# # =====================================
# # 2. (Opcional, mas recomendado)
# # Tratar pesid nulo (senão ele ignora no groupby)
# # =====================================
# df_novo["pesid"] = df_novo["pesid"].fillna(-1)

# # =====================================
# # 3. Garantir 1 linha por pessoa no acidente
# # (evita contar pessoa duplicada)
# # =====================================
# df_pessoa = (
#     df_novo
#     .groupby(["id", "pesid"], as_index=False)
#     .agg({
#         "mortos": "max",
#         "feridos_graves": "max",
#         "feridos_leves": "max",
#         "ilesos": "max"
#     })
# )

# # =====================================
# # 4. Somar por acidente
# # =====================================
# df_soma = (
#     df_pessoa
#     .groupby("id", as_index=False)
#     .agg({
#         "mortos": "sum",
#         "feridos_graves": "sum",
#         "feridos_leves": "sum",
#         "ilesos": "sum"
#     })
#     .rename(columns={
#         "mortos": "Sum_Mortos",
#         "feridos_graves": "Sum_Feridos_Graves",
#         "feridos_leves": "Sum_Feridos_Leves",
#         "ilesos": "Sum_Ilesos"
#     })
# )

# # =====================================
# # 5. Criar total de vítimas
# # =====================================
# df_soma["Sum_Total_Vitimas"] = (
#     df_soma["Sum_Mortos"] +
#     df_soma["Sum_Feridos_Graves"] +
#     df_soma["Sum_Feridos_Leves"]
# )

# # =====================================
# # 6. Trazer pro dataset original (cópia)
# # =====================================
# df_novo = df_novo.merge(df_soma, on="id", how="left")

# # =====================================
# # 7. Salvar arquivo
# # =====================================
# df_novo.to_csv(r"C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\acidentes_tratado.csv", index=False)

# # (opcional - melhor performance)
# # df_novo.to_parquet("acidentes_tratado.parquet", index=False)

# # =====================================
# # 8. Conferência rápida
# # =====================================


# print(df_novo.head())
# print(df_novo[[
#     "id",
#     "pesid",
#     "Sum_Mortos",
#     "Sum_Feridos_Graves",
#     "Sum_Feridos_Leves",
#     "Sum_Total_Vitimas"
# ]].head(10))

In [8]:
import pandas as pd

# =====================================
# 1. Carregar os datasets
# =====================================

# Dataset com as somas por (id + marca)
df_novo = pd.read_csv(
    "C:/Fause/Programas/EngSoft/repos/ML-Transportes/PBIC/acidentes_por_marca.csv"
)

# Dataset filtrado (destino)
df_filtrado = pd.read_csv(
    r"C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\Etapa3\acidentes_pbic_2020_2025_Final.csv"
)

# =====================================
# 2. Garantir tipos iguais
# =====================================

df_novo["id"] = df_novo["id"].astype(str)
df_filtrado["id"] = df_filtrado["id"].astype(str)

df_novo["marca"] = df_novo["marca"].astype(str)
df_filtrado["marca"] = df_filtrado["marca"].astype(str)

# =====================================
# 3. Selecionar colunas agregadas (id + marca)
# =====================================

colunas_sum = [
    "id",
    "marca",
    "Sum_Mortos",
    "Sum_Feridos_Graves",
    "Sum_Feridos_Leves",
    "Sum_Ilesos",
    "Sum_Total_Vitimas"
]

df_sum = (
    df_novo[colunas_sum]
    .drop_duplicates(subset=["id", "marca"])  # 🔥 chave mudou aqui
)

# =====================================
# 4. Merge correto (id + marca)
# =====================================

df_final = df_filtrado.merge(
    df_sum,
    on=["id", "marca"],  # 🔥 ESSENCIAL
    how="left"
)

# =====================================
# 5. Salvar
# =====================================

df_final.to_csv(
    r"C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\acidentes_pbic_com_somas_marca.csv",
    index=False
)

# =====================================
# 6. Conferência
# =====================================

print("Colunas finais:")
print(df_final.columns)

print("\nExemplo:")
print(df_final[[
    "id",
    "marca",
    "Sum_Mortos",
    "Sum_Feridos_Graves",
    "Sum_Feridos_Leves",
    "Sum_Total_Vitimas"
]].head(10))

C:\Users\fause\AppData\Local\Temp\ipykernel_11076\441560466.py:8: DtypeWarning: Columns (0: latitude, 1: longitude) have mixed types. Specify dtype option on import or set low_memory=False.
  df_novo = pd.read_csv(
C:\Users\fause\AppData\Local\Temp\ipykernel_11076\441560466.py:13: DtypeWarning: Columns (0: latitude, 1: longitude) have mixed types. Specify dtype option on import or set low_memory=False.
  df_filtrado = pd.read_csv(


Colunas finais:
Index(['id', 'pesid', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br',
       'km', 'municipio', 'causa_principal', 'causa_acidente',
       'ordem_tipo_acidente', 'tipo_acidente', 'classificacao_acidente',
       'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista',
       'tracado_via', 'uso_solo', 'id_veiculo', 'tipo_veiculo', 'marca',
       'ano_fabricacao_veiculo', 'tipo_envolvido', 'estado_fisico', 'idade',
       'sexo', 'ilesos', 'feridos_leves', 'feridos_graves', 'mortos',
       'latitude', 'longitude', 'regional', 'delegacia', 'uop', 'ano_arquivo',
       'mes', 'dia_mes', 'hora', 'gravidade_numerica', 'total_vitimas',
       'Marca_Principal', 'Modelo_Grupo', 'Nome_Modelo', 'Fabricante',
       'Modelo', 'Sum_Mortos', 'Sum_Feridos_Graves', 'Sum_Feridos_Leves',
       'Sum_Ilesos', 'Sum_Total_Vitimas'],
      dtype='str')

Exemplo:
         id                                         marca  Sum_Mortos  \
0  260031.0  FIAT/SIENA ATTRACTIV 1.4/

In [9]:
df1 = pd.read_csv(
    r'C:\Fause\Programas\EngSoft\repos\ML-Transportes\PBIC\acidentes_pbic_com_somas_marca.csv',
    encoding='utf-8',
    parse_dates=['data_inversa'],
    dayfirst=True,
    low_memory=False
)

print("Shape inicial:", df1.shape)


Shape inicial: (1678326, 53)
